In [17]:
import geopandas as gpd
import pandas as pd
import pandas as pd
from scipy import stats

coded:

Jung (0), Alt (1)

Männlich (1), Weiblich (2), Divers (3) 

Urban (sehr städtisch, zentrumsnah) (1), Sub-urban (Agglomeration) (2), Ländlich (3)

but:

Jung (1), Alt (0)

Männlich (1), Weiblich (2), Divers (3) 

Urban (sehr städtisch, zentrumsnah) (1), Sub-urban (Agglomeration) (2), Ländlich (3)

# read

In [ ]:
gdf = gpd.read_file("../../intermediate/active_move_respond_CH.geojson", driver='GeoJSON')

In [ ]:
gdf.head(1)

In [20]:
bl = pd.read_csv("../../raw/bl_completed.csv")

In [ ]:
bl.head(1)

In [22]:
gdf = gdf.merge(bl, on= 'PID', how = 'left')

In [23]:
ndvi = pd.read_csv("../../intermediate/tripleg_ndvi_exposure_step10m_b100.csv")

In [24]:
ndvi = ndvi[['trip_id','mean_qi']]

In [25]:
gdf = gdf.merge(ndvi, on = 'trip_id', how = 'left')

In [ ]:
gdf.columns

In [27]:
gdf = gdf[~gdf['mean_qi'].isna()]

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. Select the required columns
# =========================
demo = gdf[["PID", "AgeClass", "blsex", "bl_home_type"]].copy()
demo = demo.dropna(subset=["PID"])

# =========================
# 2. Check whether demographic values are consistent within each PID
# =========================
check = demo.groupby("PID")[["AgeClass", "blsex", "bl_home_type"]].nunique()
inconsistent = check[(check > 1).any(axis=1)]

print("Number of PIDs with inconsistent demographic info:", len(inconsistent))
# If the count is nonzero, inspect the affected PIDs
# display(inconsistent.head(20))

# =========================
# 3. Keep one row per PID to create a participant-level dataset
# =========================
demo_person = demo.drop_duplicates(subset=["PID"]).copy()

print("Number of unique persons:", demo_person["PID"].nunique())

# =========================
# 4. Map variable codes to labels
# =========================
age_map = {
    0: "Older adults",
    1: "Younger adults"
}

sex_map = {
    1: "Male",
    2: "Female",
    3: "Diverse"
}

home_map = {
    1: "Urban",
    2: "Sub-urban",
    3: "Rural"
}

demo_person["AgeGroup"] = demo_person["AgeClass"].map(age_map)
demo_person["Gender"] = demo_person["blsex"].map(sex_map)
demo_person["HomeType"] = demo_person["bl_home_type"].map(home_map)

# Preserve unrecognized codes as strings
demo_person["AgeGroup"] = demo_person["AgeGroup"].fillna(demo_person["AgeClass"].astype(str))
demo_person["Gender"] = demo_person["Gender"].fillna(demo_person["blsex"].astype(str))
demo_person["HomeType"] = demo_person["HomeType"].fillna(demo_person["bl_home_type"].astype(str))

# =========================
# 5. Define a helper function to calculate n (%)
# =========================
def make_group_summary(data, group_col, var_col, categories, variable_name):
    """
    Output columns:
    Variable | Category | Overall | Younger adults | Older adults
    """
    rows = []
    
    n_total = len(data)
    n_by_group = data[group_col].value_counts(dropna=False).to_dict()

    for cat in categories:
        row = {
            "Variable": variable_name,
            "Category": cat
        }

        # Overall
        n_overall = (data[var_col] == cat).sum()
        pct_overall = n_overall / n_total * 100 if n_total > 0 else np.nan
        row["Overall"] = f"{n_overall} ({pct_overall:.1f}%)"

        # By age group
        for grp in ["Younger adults", "Older adults"]:
            sub = data[data[group_col] == grp]
            n_grp_total = len(sub)
            n_grp = (sub[var_col] == cat).sum()
            pct_grp = n_grp / n_grp_total * 100 if n_grp_total > 0 else np.nan
            row[grp] = f"{n_grp} ({pct_grp:.1f}%)"

        rows.append(row)

    return pd.DataFrame(rows)
# =========================
# 6. Create the participant count row
# =========================
n_row = pd.DataFrame([{
    "Variable": "N",
    "Category": "Participants",
    "Overall": str(len(demo_person)),
    "Younger adults": str((demo_person["AgeGroup"] == "Younger adults").sum()),
    "Older adults": str((demo_person["AgeGroup"] == "Older adults").sum())
}])

# =========================
# 7. Create the gender summary
# =========================
sex_table = make_group_summary(
    data=demo_person,
    group_col="AgeGroup",
    var_col="Gender",
    categories=["Male",
    "Female",
  "Diverse"],
    variable_name="Gender"
)

# =========================
# 8. Create the home type summary
# =========================
home_table = make_group_summary(
    data=demo_person,
    group_col="AgeGroup",
    var_col="HomeType",
    categories=["Urban", "Sub-urban", "Rural"],
    variable_name="Home type"
)

# =========================
# 9. Combine the summaries into the final table
# =========================
demo_table = pd.concat([n_row, sex_table, home_table], ignore_index=True)

display(demo_table)

In [ ]:
gdf.columns

In [30]:
gdf[['trip_id', 'PID', 'timeEnd', 'timeStart', 'secondActivity',
       'secondActivity_other', 'duration_s', 'Rating', 'pp_other',
       'pp_acquaintances_colleagues', 'pp_family', 'pp_strangers',
       'pp_friends', 'pp_alone', 'pp_partner',  'AgeClass', 'blsex',
       'bl_home_type', 'mean_qi']].to_csv('temp_for_model_100.csv')

In [ ]:
gdf.shape